In [1]:
# Cellule 1 : imports, graine, chemins
import os, glob, json, random, unicodedata
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

SEED = 42
random.seed(SEED); np.random.seed(SEED)

hits = glob.glob("/kaggle/input/**/train.tsv", recursive=True)
assert hits, "train.tsv introuvable : ajoute ton dataset avec Add Input"
P = os.path.dirname(hits[0]) + "/"

IMG_FICHIER = "img_clip_vit_b32_%s.npy"   # embeddings image : CLIP ViT-B/32 gele (sortie du notebook 02 v2)
emb = glob.glob("/kaggle/input/**/emb/" + IMG_FICHIER % "train", recursive=True)
assert emb, ("embeddings CLIP introuvables : Add Input, onglet Notebooks (ou Your Work), choisis la version v2 de "
             "ton notebook 02_embeddings_e2_e3 (sa sortie contient emb/img_clip_vit_b32_train.npy)")
EMB = os.path.dirname(emb[0]) + "/"
assert os.path.exists(EMB + "txt_train.npy"), "txt_train.npy manquant dans " + EMB

OUT = "/kaggle/working/"
os.makedirs(OUT + "results/mlflow", exist_ok=True)
print("splits     :", P)
print("embeddings :", EMB)

splits     : /kaggle/input/datasets/sahnounsahnoun/data-fakenewes/
embeddings : /kaggle/input/notebooks/sahnounsahnoun/e2-e3-embeddings/emb/


In [2]:
# Cellule 2 : chargement des splits et des embeddings, controles d'alignement
S = ["train", "val", "test"]
df = {s: pd.read_csv(P + s + ".tsv", sep="\t") for s in S}
y = {s: df[s]["2_way_label"].values for s in S}
y6 = {s: df[s]["6_way_label"].values for s in S}

Xi = {s: np.load(EMB + IMG_FICHIER % s) for s in S}   # CLIP ViT-B/32 gele, 512 dimensions
Xt = {s: np.load(EMB + "txt_%s.npy" % s) for s in S}   # DistilBERT gele, 768 dimensions
for s in S:
    assert len(df[s]) == len(Xi[s]) == len(Xt[s]), (
        "%s : %d lignes dans le tsv, %d embeddings image, %d embeddings texte. "
        "Les embeddings viennent d'une autre version du dataset : relance 02." % (s, len(df[s]), len(Xi[s]), len(Xt[s])))
    print(s, len(df[s]), "exemples | image", Xi[s].shape, "| texte", Xt[s].shape)

train 7432 exemples | image (7432, 512) | texte (7432, 768)
val 1403 exemples | image (1403, 512) | texte (1403, 768)
test 1396 exemples | image (1396, 512) | texte (1396, 768)


In [3]:
# Cellule 3 : evaluation, entrainement (meme protocole que E2 et E3) et gain avec intervalle
GRID = [0.0003, 0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1]
GRID_ETENDUE = [0.00001, 0.00003, 0.0001, 0.0003, 0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1]   # celle du notebook 02, pour l'image

def metrics(y_true, p):
    pred = (p > 0.5).astype(int)
    return dict(acc=accuracy_score(y_true, pred),
                prec=precision_score(y_true, pred, average="macro", zero_division=0),
                rec=recall_score(y_true, pred, average="macro", zero_division=0),
                macro_f1=f1_score(y_true, pred, average="macro", zero_division=0),
                auc=roc_auc_score(y_true, p))

def run_lr(name, X, grid=None):
    grid = grid or GRID
    mk = lambda C: make_pipeline(StandardScaler(), LogisticRegression(
        C=C, max_iter=3000, class_weight="balanced", random_state=SEED))
    # choix de C sur la validation uniquement
    val = {C: f1_score(y["val"], mk(C).fit(X["train"], y["train"]).predict(X["val"]), average="macro")
           for C in grid}
    best = max(val, key=val.get)
    print(name, "Macro F1 validation par C :", {c: round(v, 4) for c, v in val.items()})
    if best in (grid[0], grid[-1]):
        print("attention : le meilleur C est en bord de grille")
    clf = mk(best).fit(X["train"], y["train"])
    pv = clf.predict_proba(X["val"])[:, 1]
    pt = clf.predict_proba(X["test"])[:, 1]
    m = metrics(y["test"], pt)
    print(name, "meilleur C =", best, "| val F1 =", round(val[best], 4),
          "| TEST", {k: round(float(v), 4) for k, v in m.items()})
    np.save(OUT + "results/" + name.lower() + "_val_proba.npy", pv)
    np.save(OUT + "results/" + name.lower() + "_test_proba.npy", pt)
    return dict(best_C=best, val_macro_f1=val[best], val=metrics(y["val"], pv), test=m), pv, pt

def gain_points(y_true, p_fus, p_ref, B=1000):
    """Gain de Macro F1 (fusion - reference) en points, avec intervalle a 95 % par bootstrap sur le test."""
    rng = np.random.default_rng(SEED)
    pf, pr = (p_fus > 0.5).astype(int), (p_ref > 0.5).astype(int)
    g = 100 * (f1_score(y_true, pf, average="macro") - f1_score(y_true, pr, average="macro"))
    n, d = len(y_true), []
    for _ in range(B):
        i = rng.integers(0, n, n)
        d.append(100 * (f1_score(y_true[i], pf[i], average="macro") - f1_score(y_true[i], pr[i], average="macro")))
    return g, tuple(np.percentile(d, [2.5, 97.5]))

In [4]:
# Cellule 4 : modeles de reference E2 (texte) et E3d (image CLIP), et controle
res, pval, ptest = {}, {}, {}
res["E2"], pval["E2"], ptest["E2"] = run_lr("E2", Xt)
res["E3d"], pval["E3d"], ptest["E3d"] = run_lr("E3d", Xi, grid=GRID_ETENDUE)

# controle : on doit retrouver tes chiffres (Macro F1 test : E2 = 0,801 de la diapo ; E3d = 0,774 du notebook 02)
for k, ref in (("E2", 0.801), ("E3d", 0.774)):
    ecart = res[k]["test"]["macro_f1"] - ref
    if abs(ecart) > 0.01:
        print("ATTENTION : %s s'ecarte de %+.3f par rapport a la reference (%.3f). Verifie que les embeddings "
              "et le dataset sont ceux de 02 avant d'interpreter." % (k, ecart, ref))
    else:
        print("Controle %s : ecart avec la reference = %+.3f, OK." % (k, ecart))

# meilleur modele seul, choisi sur la VALIDATION : c'est la reference du gain
REF = max(["E2", "E3d"], key=lambda k: res[k]["val_macro_f1"])
print("Reference du gain (meilleur modele seul sur la validation) :", REF)

E2 Macro F1 validation par C : {0.0003: 0.796, 0.001: 0.8018, 0.003: 0.8068, 0.01: 0.8068, 0.03: 0.8082, 0.1: 0.8104, 0.3: 0.8075, 1: 0.8096}
E2 meilleur C = 0.1 | val F1 = 0.8104 | TEST {'acc': 0.8009, 'prec': 0.8009, 'rec': 0.8008, 'macro_f1': 0.8008, 'auc': 0.8615}
E3d Macro F1 validation par C : {1e-05: 0.7406, 3e-05: 0.749, 0.0001: 0.7663, 0.0003: 0.7812, 0.001: 0.7849, 0.003: 0.7843, 0.01: 0.7788, 0.03: 0.7717, 0.1: 0.7732, 0.3: 0.7711, 1: 0.7675}
E3d meilleur C = 0.001 | val F1 = 0.7849 | TEST {'acc': 0.7744, 'prec': 0.7754, 'rec': 0.7742, 'macro_f1': 0.7741, 'auc': 0.8591}
Controle E2 : ecart avec la reference = -0.000, OK.
Controle E3d : ecart avec la reference = +0.000, OK.
Reference du gain (meilleur modele seul sur la validation) : E2


In [5]:
# Cellule 5 : E5b, fusion tardive = moyenne ponderee des probabilites de E2 (texte) et E3d (image CLIP)
POIDS = [round(float(w), 1) for w in np.arange(0, 1.01, 0.1)]
fuse = lambda w, p_txt, p_img: w * p_txt + (1 - w) * p_img

val_f1 = {w: f1_score(y["val"], (fuse(w, pval["E2"], pval["E3d"]) > 0.5).astype(int), average="macro")
          for w in POIDS}
w_best = max(val_f1, key=val_f1.get)
print("Macro F1 validation selon le poids du texte w :", {w: round(v, 4) for w, v in val_f1.items()})
if w_best in (0.0, 1.0):
    print("attention : w = %.1f, la fusion ignore une des deux modalites : le gain est nul par construction." % w_best)

pv5 = fuse(w_best, pval["E2"], pval["E3d"])
pt5 = fuse(w_best, ptest["E2"], ptest["E3d"])
res["E5b"] = dict(poids_texte=w_best, val_macro_f1=val_f1[w_best], val=metrics(y["val"], pv5), test=metrics(y["test"], pt5))
pval["E5b"], ptest["E5b"] = pv5, pt5
np.save(OUT + "results/e5b_val_proba.npy", pv5)
np.save(OUT + "results/e5b_test_proba.npy", pt5)
print("E5b : w =", w_best, "| val F1 =", round(val_f1[w_best], 4),
      "| TEST", {k: round(float(v), 4) for k, v in res["E5b"]["test"].items()})

# controle sans aucun reglage : moyenne simple (w = 0,5)
moy = metrics(y["test"], fuse(0.5, ptest["E2"], ptest["E3d"]))
print("Controle, moyenne simple (w = 0,5) : TEST", {k: round(float(v), 4) for k, v in moy.items()})

FUS = "E5b"
FUS_LIB = "E5b fusion tardive (moyenne ponderee, CLIP)"
ETAPE, NOM = "E5b", "fusion_tardive_moyenne_ponderee_clip"
PARAMS_FUS = dict(entree="titre + image",
                  representation="probabilites de E2 (DistilBERT gele) et E3d (CLIP ViT-B/32 gele)",
                  classifieur="p = w*p_texte + (1-w)*p_image, w choisi sur la validation",
                  poids_texte=w_best, pas_de_poids=0.1)
EXTRA_MET = dict(moyenne_simple_accuracy=moy["acc"], moyenne_simple_f1_macro=moy["macro_f1"], moyenne_simple_auc=moy["auc"])

Macro F1 validation selon le poids du texte w : {0.0: 0.7849, 0.1: 0.8101, 0.2: 0.8301, 0.3: 0.8309, 0.4: 0.8502, 0.5: 0.8424, 0.6: 0.8296, 0.7: 0.8289, 0.8: 0.8232, 0.9: 0.8168, 1.0: 0.8104}
E5b : w = 0.4 | val F1 = 0.8502 | TEST {'acc': 0.8431, 'prec': 0.8438, 'rec': 0.843, 'macro_f1': 0.843, 'auc': 0.9084}
Controle, moyenne simple (w = 0,5) : TEST {'acc': 0.8345, 'prec': 0.8348, 'rec': 0.8344, 'macro_f1': 0.8345, 'auc': 0.9061}


In [6]:
# Cellule 6 : tableau comparatif, gain face au meilleur modele seul, precision par type
n_test = len(y["test"])
lignes = {
    "Majoritaire (tout vrai)": metrics(y["test"], np.ones(n_test)),
    "E2 texte DistilBERT": res["E2"]["test"],
    "E3d image CLIP ViT-B/32": res["E3d"]["test"],
    FUS_LIB: res[FUS]["test"],
}
tab = pd.DataFrame(lignes).T[["acc", "macro_f1", "auc"]].astype(float)
tab["marge_acc_95"] = 1.96 * np.sqrt(tab["acc"] * (1 - tab["acc"]) / n_test)
print("Test (%d exemples) ; marge_acc_95 = demi-largeur de l'intervalle a 95 %% de l'exactitude :" % n_test)
print(tab.round(4), "\n")

g, (lo, hi) = gain_points(y["test"], ptest[FUS], ptest[REF])
g_acc = 100 * (res[FUS]["test"]["acc"] - res[REF]["test"]["acc"])
print("GAIN de %s par rapport a %s (meilleur modele seul) :" % (FUS, REF))
print("  Macro F1 : %+.1f points, intervalle a 95 %% [%+.1f ; %+.1f]" % (g, lo, hi))
print("  exactitude : %+.1f points" % g_acc)
if lo <= 0 <= hi:
    print("  L'intervalle contient 0 : le gain n'est pas etabli sur ce test (une seule graine).")
elif lo > 0:
    print("  L'intervalle est au-dessus de 0 : gain positif sur ce test (une seule graine, a confirmer avec d'autres graines).")
else:
    print("  L'intervalle est en dessous de 0 : la fusion fait moins bien que le meilleur modele seul sur ce test.")

# correspondance des types a 6 classes : a verifier (diagnostic en cours)
names = {0: "vrai", 1: "satire", 2: "trompeur", 3: "bots", 4: "image manipulee", 5: "fausse connexion"}
per_type = pd.DataFrame({"n": pd.Series(y6["test"]).value_counts().sort_index()})
for k in ["E2", "E3d", FUS]:
    ok = (ptest[k] > 0.5).astype(int) == y["test"]
    per_type[k] = pd.Series(ok).groupby(y6["test"]).mean().round(3)
per_type["%s-%s" % (FUS, REF)] = (per_type[FUS] - per_type[REF]).round(3)
per_type.index = [str(i) + " " + names.get(i, "?") for i in per_type.index]
print("\nPrecision par type (test) :")
print(per_type)

Test (1396 exemples) ; marge_acc_95 = demi-largeur de l'intervalle a 95 % de l'exactitude :
                                                acc  macro_f1     auc  \
Majoritaire (tout vrai)                      0.5029    0.3346  0.5000   
E2 texte DistilBERT                          0.8009    0.8008  0.8615   
E3d image CLIP ViT-B/32                      0.7744    0.7741  0.8591   
E5b fusion tardive (moyenne ponderee, CLIP)  0.8431    0.8430  0.9084   

                                             marge_acc_95  
Majoritaire (tout vrai)                            0.0262  
E2 texte DistilBERT                                0.0209  
E3d image CLIP ViT-B/32                            0.0219  
E5b fusion tardive (moyenne ponderee, CLIP)        0.0191   

GAIN de E5b par rapport a E2 (meilleur modele seul) :
  Macro F1 : +4.2 points, intervalle a 95 % [+2.4 ; +6.0]
  exactitude : +4.2 points
  L'intervalle est au-dessus de 0 : gain positif sur ce test (une seule graine, a confirmer avec d'au

In [7]:
# Cellule 7 : fichier de resultats pour MLflow et metriques completes
def slug(t):
    return "_".join(unicodedata.normalize("NFKD", t).encode("ascii", "ignore").decode().lower().split())

m, v = res[FUS]["test"], res[FUS]["val"]
met = dict(accuracy=m["acc"], f1_macro=m["macro_f1"], auc=m["auc"],
           val_accuracy=v["acc"], val_f1_macro=v["macro_f1"], val_auc=v["auc"],
           gain_f1_points=g, gain_f1_ic95_bas=lo, gain_f1_ic95_haut=hi, gain_acc_points=g_acc)
met.update(EXTRA_MET)
ok = (ptest[FUS] > 0.5).astype(int) == y["test"]
for t in sorted(np.unique(y6["test"])):
    masque = y6["test"] == t
    met["acc_type_" + slug(names.get(int(t), str(t)))] = float(ok[masque].mean())
    met["n_type_" + slug(names.get(int(t), str(t)))] = int(masque.sum())

AVERT = ("Types a 6 classes : correspondance des etiquettes en cours de verification ; "
         "fausse_connexion : peu de publications ; bots : tres peu.")
params = dict(PARAMS_FUS, reference_gain=REF, n_train=len(df["train"]), n_val=len(df["val"]), n_test=len(df["test"]))
out = dict(etape=ETAPE, nom=NOM, params=params, metrics={k: float(x) for k, x in met.items()},
           tags=dict(source="notebook Kaggle " + '06b_e5b_fusion_tardive_clip', avertissement_types=AVERT))
with open(OUT + "results/mlflow/resultat_%s_%s.json" % (ETAPE, NOM), "w", encoding="utf-8") as f:
    json.dump(out, f, indent=2, ensure_ascii=False)

log = dict(seed=SEED, n={s: len(df[s]) for s in S}, reference_gain=REF, gain_f1_points=g,
           gain_f1_ic95=[lo, hi], res=res, table=tab.to_dict(), per_type=per_type.to_dict())
with open(OUT + "results/%s_metrics.json" % ETAPE.lower(), "w", encoding="utf-8") as f:
    json.dump(log, f, indent=2, ensure_ascii=False,
              default=lambda o: o.item() if hasattr(o, "item") else str(o))

for root, _, files in os.walk(OUT):
    for fn in sorted(files):
        p = os.path.join(root, fn)
        print(p, round(os.path.getsize(p) / 1e6, 2), "Mo")

/kaggle/working/__notebook__.ipynb 0.02 Mo
/kaggle/working/results/e2_test_proba.npy 0.01 Mo
/kaggle/working/results/e2_val_proba.npy 0.01 Mo
/kaggle/working/results/e3d_test_proba.npy 0.01 Mo
/kaggle/working/results/e3d_val_proba.npy 0.01 Mo
/kaggle/working/results/e5b_metrics.json 0.0 Mo
/kaggle/working/results/e5b_test_proba.npy 0.01 Mo
/kaggle/working/results/e5b_val_proba.npy 0.01 Mo
/kaggle/working/results/mlflow/resultat_E5b_fusion_tardive_moyenne_ponderee_clip.json 0.0 Mo
